# Extracting Structured Data from Unstructured Text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/StructuredDataExtraction/StructuredDataExtraction.ipynb)

Data sources like invoices, emails, and support tickets rarely arrive in clean, machine-readable formats. Traditional extraction methods rely on regular expressions and custom JSON parsers, which are difficult to maintain and frequently fail when document layouts change.

This guide demonstrates an alternative approach. By defining the target data structure as a typed Python schema, Mellea manages prompt construction, model execution, schema validation, and automatic retries.

**What you will learn:**

- Limitations of regex and manual JSON parsing for text extraction
- Using `@generative` stubs to map Pydantic schemas to extraction functions
- Using `instruct(format=...)` for dynamic prompts and structured output
- Implementation patterns for invoices, emails, and support tickets
- Applying `requirements` for domain constraints and error recovery

**Prerequisites:** Ollama running locally with `granite4.1:3b` pulled, **or** an OpenAI-compatible API key (see *Backend Setup* below).

## Install dependencies

In [ ]:
%pip install -q git+https://github.com/ibm-granite-community/utils
%pip install -q mellea pydantic

## Backend setup

This notebook runs IBM Granite locally via Ollama. The cells below install Ollama,
start the server, and pull the model, no API keys required.

In [ ]:
# Install Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model
!ollama pull granite4.1:8b

In [ ]:
import os
from mellea import start_session

os.environ["LITELLM_MODEL"] = "ollama/granite4.1:8b"

m = start_session()
print("Session ready:", m)

## Traditional extraction: regex limitations

Before implementing LLM-based solutions, it is useful to review the standard regex approach. The following code shows a typical regex-based extractor for invoice data:

In [ ]:
import re

def extract_invoice_fragile(text: str) -> dict:
    """Traditional regex approach — breaks on format variation."""
    invoice_no = re.search(r'Invoice\s*#?\s*([\w-]+)', text)
    total      = re.search(r'Total[:\s]+\$?([\d,\.]+)', text)
    due_date   = re.search(r'Due[:\s]+(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})', text)
    return {
        "invoice_number": invoice_no.group(1) if invoice_no else None,
        "total":          total.group(1)      if total      else None,
        "due_date":       due_date.group(1)   if due_date   else None,
    }

# Works on a clean, predictable format:
clean = "Invoice #INV-2024-089  Total: $1,450.00  Due: 05/30/2024"
print("Clean: ", extract_invoice_fragile(clean))

# Fails on real-world variation:
messy_invoice = """
    Ref: 2024-089 | Mao-Kwikowski Mercantile
    Issued 3rd March 2024 — Payment by: 30 May 2024

    Items ordered:
      - Widget Pro x 5 @ 150 USD each
      - Mounting Kit x 2 @ 50 USD each
      - Express Shipping (flat fee): 100 USD

    Amount due: USD 950 (excl. VAT)
    Bank transfer to: Tycho Bank, IBAN XX00 0000 0000
"""
print("Messy:", extract_invoice_fragile(messy_invoice))

The regex fails to extract any fields from the unstructured version. In production environments, invoices originate from multiple vendors with varying layouts. Maintaining distinct regular expressions for each variation is not scalable.

The declarative approach allows you to define the schema once and delegate format normalization to the model.

## Pattern 1: Using `@generative` stubs with Pydantic schemas

A `@generative` stub is a body-less Python function. The decorator translates the function signature into a structured LLM call based on three components:

- **Docstring:** The task description (system prompt)
- **Parameter names:** The input variables provided to the model
- **Return annotation:** The Pydantic schema enforcing the output structure

The returned object is guaranteed to be a validated instance of the specified type. If the model generates invalid output, Mellea automatically initiates a retry sequence.

### Example: Parsing an unstructured invoice

In [ ]:
from typing import List, Optional
from pydantic import BaseModel
from mellea import generative


class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    invoice_number: str
    vendor_name: str
    issue_date: Optional[str] = None   # ISO 8601 when present
    due_date: Optional[str] = None
    line_items: List[LineItem]
    total_amount: float
    currency: str


@generative
def parse_invoice(text: str) -> Invoice:
    """Parse the invoice text and extract all fields.
    For each line item, extract description as product name only (without quantity or price),
    quantity as integer, and unit_price as float.
    Convert all dates to ISO 8601 format (YYYY-MM-DD).
    If a field is not present in the text, return null."""
    ...

invoice = parse_invoice(m, text=messy_invoice)
print(invoice.model_dump_json(indent=2))

The `parse_invoice` function processes varying invoice layouts, date formats, and currency symbols without requiring code modifications.

The extracted fields are accessible directly on the typed object:

In [ ]:
print(f"Invoice:  {invoice.invoice_number}")
print(f"Vendor:   {invoice.vendor_name}")
print(f"Due:      {invoice.due_date}")
print(f"Total:    {invoice.total_amount} {invoice.currency}")
for item in invoice.line_items:
    print(f"  {item.quantity}x {item.description} @ {item.unit_price}")

### Example: Extracting contact details from an email thread

Email threads often contain quoted headers, inline replies, and signature blocks. A `@generative` stub isolates the requested entities and ignores irrelevant text.

In [ ]:
from typing import Optional
from pydantic import BaseModel
from mellea import generative


class ContactRecord(BaseModel):
    full_name: str
    email: str
    phone: Optional[str] = None
    company: Optional[str] = None
    role: Optional[str] = None


@generative
def extract_contact(text: str) -> ContactRecord:
    """Extract the primary sender's contact information from the email thread.
    Focus on the most recent message. Extract email address from the From: header.
    Extract company name separately from job role.
    Return null for any field not present."""
    ...


messy_email = """
From: j.morrison@protogencorp.com
Sent: Mon, 5 May 2026 09:12:33 +0000
Subject: Re: Re: Fwd: Q2 partnership enquiry

Hi there,

Following up on our earlier thread. I'm the Head of Business Development
at Protogen Corp. Feel free to reply here.

Best,
James Morrison

---
> On 3 May 2026, sales@vendorco.com wrote:
> Thanks for the interest, we'll follow up shortly.
"""

contact = extract_contact(m, text=messy_email)
print(f"Name:    {contact.full_name}")
print(f"Email:   {contact.email}")
print(f"Phone:   {contact.phone}")
print(f"Company: {contact.company}")
print(f"Role:    {contact.role}")

## Pattern 2: Using `instruct(format=...)` for dynamic execution

While `@generative` is suited for static, reusable extraction logic, `instruct(format=...)` is designed for dynamic prompt construction, such as injecting routing rules, document context, or runtime variables.

The `format` parameter accepts a Pydantic model class to enforce the schema during generation. The output can then be parsed using `Model.model_validate_json()`.

### Example: Support ticket triage with runtime routing rules

In [ ]:
from typing import List, Literal
from pydantic import BaseModel


class TicketSummary(BaseModel):
    issue_type: Literal["billing", "technical", "account", "feature_request", "other"]
    severity: Literal["low", "medium", "high", "critical"]
    affected_components: List[str]
    suggested_team: str
    one_line_summary: str


# Routing rules loaded at runtime — a @generative stub cannot hold them.
# instruct() with grounding_context is the appropriate pattern here.
routing_policy = """
- billing issues       → Finance team
- critical technical   → On-call engineering
- account issues       → Customer Success
- feature requests     → Product team
"""

messy_ticket = """
[10:02] Customer: hi my dashboard wont load since this mornings update
[10:03] Agent: Which browser?
[10:03] Customer: chrome 124. also the csv export just spins forever
[10:04] Agent: Dashboard bug is known in v2.3.1, hotfix rolling out.
                CSV export is new, can you try incognito?
[10:05] Customer: same in incognito. email notifs also stopped 2 days ago
[10:06] Agent: Escalating CSV and email issues to engineering.
[10:07] Customer: ok thanks, pretty frustrating tbh
"""

result = m.instruct(
    "Analyse the support chat (ticket) and classify it using the routing policy (policy). "
    "List ALL affected components mentioned in the conversation, including dashboard, csv export, and email notifications.",
    grounding_context={
        "ticket": messy_ticket,
        "policy": routing_policy,
    },
    format=TicketSummary,
)

summary = TicketSummary.model_validate_json(str(result))
print(f"Type:       {summary.issue_type}")
print(f"Severity:   {summary.severity}")
print(f"Components: {summary.affected_components}")
print(f"Route to:   {summary.suggested_team}")
print(f"Summary:    {summary.one_line_summary}")

The routing policy is passed at execution time via `grounding_context`. Updating the policy dictionary applies new rules to subsequent calls without modifying the core extraction logic.

### When to use each pattern

| | `@generative` stub | `instruct(format=...)` |
|---|---|---|
| Reusable, named function | ✅ | — |
| Dynamic prompt / grounding context | — | ✅ |
| Direct attribute access on result | ✅ | needs `model_validate_json` |
| IDE type-checking | ✅ | — |

## Pattern 3: Applying `requirements` for domain constraints

The requirements system enforces domain-specific rules on the model's output. If a constraint is violated, the instruct-validate-repair (IVR) loop prompts the model to correct the error, up to a defined retry budget.

Constraints can be implemented in two ways:
- **Semantic constraint:** A natural language string evaluated by the model.
- **`validation_fn`:** A deterministic Python function (e.g., for regex, bounds checking, or length validation).

### Example: Enforcing ISO dates and positive numerical values

In [ ]:
import re
from typing import Optional
from pydantic import BaseModel, ValidationError
from mellea import generative
from mellea.stdlib.requirements import req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy


class ValidatedInvoice(BaseModel):
    invoice_number: str
    vendor_name: str
    due_date: str
    total_amount: float
    currency: str


def _iso_date_in_json(text: str) -> tuple[bool, str]:
    """Verify that due_date in the JSON output is YYYY-MM-DD."""
    try:
        parsed = ValidatedInvoice.model_validate_json(text)
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", parsed.due_date):
            return (True, "")
        return (False, f"due_date '{parsed.due_date}' is not ISO 8601 (YYYY-MM-DD).")
    except ValidationError as e:
        return (False, str(e))


def _positive_total(text: str) -> tuple[bool, str]:
    """Verify that total_amount is greater than zero."""
    try:
        parsed = ValidatedInvoice.model_validate_json(text)
        if parsed.total_amount > 0:
            return (True, "")
        return (False, "total_amount must be greater than zero.")
    except ValidationError as e:
        return (False, str(e))


@generative
def parse_invoice_validated(text: str) -> ValidatedInvoice:
    """Parse the invoice and extract vendor, invoice number, due date,
    total amount, and currency. Convert the due date to ISO 8601."""
    ...


invoice_v2 = parse_invoice_validated(
    m,
    text=messy_invoice,
    requirements=[
        req(
            "The due_date field must be ISO 8601: YYYY-MM-DD.",
            validation_fn=simple_validate(_iso_date_in_json),
        ),
        req(
            "The total_amount field must be a positive number.",
            validation_fn=simple_validate(_positive_total),
        ),
    ],
    strategy=RejectionSamplingStrategy(loop_budget=3),
)
print(invoice_v2.model_dump_json(indent=2))

If the model generates a non-compliant date format (e.g., `"30 May 2024"`), `_iso_date_in_json` detects the error. The IVR loop then submits a repair prompt containing the failure reason. The function guarantees either a valid `ValidatedInvoice` object or an exception if the retry budget is exceeded.

## Pattern 4: Composing extraction pipelines

Since `@generative` stubs operate as standard Python functions, they can be composed seamlessly. The following example demonstrates a three-step triage pipeline: classifying the ticket, determining escalation logic, and drafting an initial response.

In [ ]:
from typing import Literal, List
from pydantic import BaseModel
from mellea import generative


class TicketClassification(BaseModel):
    issue_type: Literal["billing", "technical", "account", "other"]
    severity: Literal["low", "medium", "high", "critical"]
    affected_components: List[str]


@generative
def classify_ticket(ticket: str) -> TicketClassification:
    """Classify the support ticket by issue type, severity, and affected components."""
    ...


@generative
def should_escalate(classification: str) -> Literal["yes", "no"]:
    """Given a JSON ticket classification, decide if it needs immediate
    escalation to a senior engineer. Answer 'yes' or 'no'."""
    ...


@generative
def draft_first_response(ticket: str, escalated: bool) -> str:
    """Draft a polite, empathetic first response to the customer.
    If escalated is True, acknowledge that a senior engineer will follow up."""
    ...


def triage_ticket(m, raw_ticket: str) -> dict:
    classification = classify_ticket(m, ticket=raw_ticket)
    escalate = should_escalate(
        m, classification=classification.model_dump_json()
    ) == "yes"
    response = draft_first_response(m, ticket=raw_ticket, escalated=escalate)
    return {
        "classification": classification.model_dump(),
        "escalate":       escalate,
        "draft_response": str(response),
    }


result = triage_ticket(m, messy_ticket)

print("=== Classification ===")
print(f"  Type:       {result['classification']['issue_type']}")
print(f"  Severity:   {result['classification']['severity']}")
print(f"  Components: {result['classification']['affected_components']}")
print(f"  Escalate:   {result['escalate']}")
print()
print("=== Draft Response ===")
print(result["draft_response"])

Each stub functions as an independent, testable unit. Control flow, including sequencing and conditional branching, is handled using standard Python syntax without requiring additional framework abstraction.

## Summary

| Pattern | API | Best for |
|---|---|---|
| Reusable typed extractor | `@generative` stub + Pydantic | Invoices, contacts, stable schemas |
| Dynamic prompt + structured output | `instruct(format=...)` | Routing rules, RAG, runtime context |
| Format / semantic enforcement | `requirements` + `validation_fn` | Dates, ranges, compliance rules |
| Multi-step extraction pipeline | Composed `@generative` stubs | Triage, enrichment, routing |

Core concepts for declarative extraction:

- **Schema as contract:** Define the target structure; the framework handles extraction and validation.
- **Docstring as instruction:** Prompts are localized within function definitions rather than external templates.
- **Native Python composition:** Extraction steps are chained using standard language features.

**Next steps:**

- [Instruct, Validate, Repair](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/InstructValidateRepair/InstructValidateRepair.ipynb): Detailed overview of the IVR loop.
- [Enforce Structured Output](https://docs.mellea.ai/how-to/enforce-structured-output): Token-level constrained decoding.
- [The Requirements System](https://docs.mellea.ai/concepts/requirements-system): Reference for declarative constraints.